Make an animation of all synCP contact maps

In [1]:
import numpy as np
import matplotlib.pyplot as plt

def do_apc(x, rm=1):
  '''given matrix do apc correction'''
  # trying to remove different number of components
  # rm=0 remove none
  # rm=1 apc
  x = np.copy(x)
  if rm == 0:
    return x
  elif rm == 1:
    a1 = x.sum(0,keepdims=True)
    a2 = x.sum(1,keepdims=True)
    y = x - (a1*a2)/x.sum()
  else:
    # decompose matrix, rm largest(s) eigenvectors
    u,s,v = np.linalg.svd(x)
    y = s[rm:] * u[:,rm:] @ v[rm:,:]
  np.fill_diagonal(y,0)
  return y
    
def get_contacts(x, symm=True, center=True, rm=1):
  # convert jacobian (L,A,L,A) to contact map (L,L)
  j = x.copy()
  if center:
    for i in range(4): j -= j.mean(i,keepdims=True)
  j_fn = np.sqrt(np.square(j).sum((1,3)))
  np.fill_diagonal(j_fn,0)
  j_fn_corrected = do_apc(j_fn, rm=rm)
  if symm:
    j_fn_corrected = (j_fn_corrected + j_fn_corrected.T)/2
  return j_fn_corrected

In [2]:
PDB_ID='1OYA'

In [9]:
for i in range(135):
    jac = np.load(f"catjac_outputs/{PDB_ID}_CP_{i}_ESM2_650M_CatJac.npy")
    for k in range(4): jac -= jac.mean(k, keepdims=True)
    jac = (jac + jac.transpose(2,3,0,1))/2
    jac = get_contacts(jac)
    plt.figure(figsize=(10,5))
    plt.title(f"{PDB_ID} CP {i}")
    plt.imshow(jac)
    plt.savefig(f"catjac_outputs/{PDB_ID}_CP_{i}_ESM2_650M_CatJac.png")
    plt.close()

ValueError: cannot reshape array of size 0 into shape (399,20,399,20)

In [10]:
PDB_ID

'1OYA'

## CODE FOR GENERATING ANIMATION GIF

In [13]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation

fig, ax = plt.subplots(figsize=(10,5))
frames = []
for i in range(134):
    jac = np.load(f"catjac_outputs/{PDB_ID}_CP_{i}_ESM2_650M_CatJac.npy")
    for k in range(4): jac -= jac.mean(k, keepdims=True)
    jac = (jac + jac.transpose(2,3,0,1))/2
    jac = get_contacts(jac)
    im = ax.imshow(jac, animated=True)
    title = ax.text(0.5, 1.01, f"{PDB_ID} CP {i}", transform=ax.transAxes, ha="center")
    frames.append([im, title])

ani = animation.ArtistAnimation(fig, frames, interval=100, blit=True)
ani.save("catjac_outputs/1OYA_CP_ESM2_650M_133.gif", writer="pillow")
plt.close()